# Train Ensemble² — 12-Channel Early Fusion Model

This notebook trains the single 12-channel model that concatenates
RGB + Lab + HSV + YCrCb as its input (Ensemble² in the paper).

**Prerequisites:** `retinal_colab_data_bundle.zip` must exist in your Drive.
The zip is extracted to Colab's local disk for fast I/O during training.
The final model checkpoint and predictions are written back to Drive.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. User Settings

Replace every `CHANGE_ME` value before running.

In [ ]:
from pathlib import Path

# Full path to the zip file on Drive
DRIVE_ZIP_PATH = Path('/content/drive/MyDrive/retinal_colab_data_bundle.zip')

# Repository
REPO_URL = 'https://github.com/mehmetaytugyuruk/retinal-color-transfer.git'
REPO_REF = 'main'

# Drive folder where the trained model will be saved
DRIVE_MODEL_ROOT = Path('/content/drive/MyDrive/retinal-color-transfer-artifacts/models')

# Training hyperparameters (keep consistent with main study)
EPOCHS          = 80
BATCH_SIZE      = 32
NUM_WORKERS     = 2
MIXED_PRECISION = 'fp16'
SEED            = 42

# Local Colab paths (fast SSD — no need to change)
LOCAL_DATA_DIR  = Path('/content/local_data')
REPO_DIR        = Path('/content/retinal-color-transfer')
LOCAL_MODEL_DIR = Path('/content/fusion_model')
MODEL_NAME      = 'rgb_lab_hsv_ycrcb_seed42'

print('Settings OK')

## 3. Validate Settings

In [ ]:
if not DRIVE_ZIP_PATH.is_file():
    raise FileNotFoundError(f'Zip not found: {DRIVE_ZIP_PATH}')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required. Go to Runtime → Change runtime type → GPU.')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Zip: {DRIVE_ZIP_PATH} ({DRIVE_ZIP_PATH.stat().st_size / 1e9:.1f} GB)')

## 4. Copy Zip to Local Disk and Extract

Colab's local SSD is ~5× faster than Drive for random reads.
We copy the zip once and extract only the four cache families needed.
The zip uses `base_rgb` for the RGB cache; we create a symlink so the
training script sees the expected `rgb/rgb/` path.


In [ ]:
import shutil, time, zipfile

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
local_zip = LOCAL_DATA_DIR / 'bundle.zip'

if not local_zip.is_file():
    print('Copying zip from Drive to local disk...')
    t0 = time.time()
    shutil.copy2(DRIVE_ZIP_PATH, local_zip)
    print(f'Copied in {time.time()-t0:.0f}s')
else:
    print('Zip already on local disk, skipping copy.')

# Mapping: script name -> actual dir name inside zip
CACHE_FLAT   = LOCAL_DATA_DIR / 'caches'
CACHE_NEEDED = {
    'rgb':   'base_rgb',   # zip uses base_rgb for the RGB cache
    'lab':   'lab',
    'hsv':   'hsv',
    'ycrcb': 'ycrcb',
}

already = all((CACHE_FLAT / src).is_dir() for src in CACHE_NEEDED.values())
if not already:
    print('Extracting cache directories...')
    prefixes = [f'caches/{src}/' for src in CACHE_NEEDED.values()]
    t0 = time.time()
    with zipfile.ZipFile(local_zip) as zf:
        members = [n for n in zf.namelist()
                   if any(n.startswith(p) for p in prefixes)]
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(LOCAL_DATA_DIR, members=members)
    print(f'Extracted {len(members)} files in {time.time()-t0:.0f}s')
else:
    print('Cache directories already extracted.')

# Build nested symlink structure that the training script expects:
#   caches_nested/<rep>/<rep>/  ->  caches/<src_name>/
CACHE_ROOT = LOCAL_DATA_DIR / 'caches_nested'
for rep, src_name in CACHE_NEEDED.items():
    nested = CACHE_ROOT / rep / rep
    nested.parent.mkdir(parents=True, exist_ok=True)
    if not nested.exists():
        nested.symlink_to(CACHE_FLAT / src_name)

print(f'\nCache root: {CACHE_ROOT}')
for rep in CACHE_NEEDED:
    count = len(list((CACHE_ROOT / rep / rep).glob('*.png')))
    print(f'  {rep}/{rep}: {count} images')

## 5. Clone and Install Repository

In [ ]:
import importlib, os, shutil, subprocess, sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--quiet', '--depth', '1',
     '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)],
    check=True,
)

src_dir = str(REPO_DIR / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
importlib.invalidate_caches()
os.chdir(REPO_DIR)

import retinal_color_transfer
print('Repo:', Path.cwd())
print('Package:', retinal_color_transfer.__file__)

## 6. Train the 12-Channel Fusion Model

The script:
- Reads RGB (`base_rgb`), Lab, HSV, YCrCb caches and concatenates into 12-ch tensors
- Initialises ResNet-50 with conv1 adapted to 12 channels (pretrained weights ×4, scaled ×1/4)
- Trains with the same protocol as all other study models (80 ep, AdamW, LDS, etc.)
- Resumes automatically if `latest_checkpoint.pt` is present
- Saves `best_checkpoint.pt`, `training_history.csv`, and `predictions.csv`


In [ ]:
import subprocess, sys

LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable, 'scripts/train_fusion12ch.py',
        '--cache-root',      str(CACHE_ROOT),
        '--model-dir',       str(LOCAL_MODEL_DIR),
        '--device',          'cuda',
        '--epochs',          str(EPOCHS),
        '--batch-size',      str(BATCH_SIZE),
        '--num-workers',     str(NUM_WORKERS),
        '--seed',            str(SEED),
        '--mixed-precision', MIXED_PRECISION,
        '--allow-weight-download',
    ],
    check=True,
)

## 7. Save Model Artifacts to Drive

In [ ]:
import pandas as pd

drive_dest = DRIVE_MODEL_ROOT / 'fusion' / MODEL_NAME
drive_dest.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {'best_checkpoint.pt', 'training_history.csv', 'predictions.csv'}
missing = EXPECTED_FILES - {f.name for f in LOCAL_MODEL_DIR.iterdir()}
if missing:
    raise RuntimeError(f'Training did not produce expected files: {missing}')

for fname in EXPECTED_FILES:
    shutil.copy2(LOCAL_MODEL_DIR / fname, drive_dest / fname)
    print(f'Saved: {drive_dest / fname}')

preds = pd.read_csv(drive_dest / 'predictions.csv')
for split in ['validation', 'test']:
    mae = preds[preds['split'] == split]['absolute_error'].mean()
    print(f'  Ensemble² {split} MAE: {mae:.4f}')

print('\nDone! Model saved to Drive.')

---
## 8. Train Ensemble²-v2 — Cosine Warmup Schedule (120 epochs)

Same as above but with a **LinearWarmup (5 ep) + CosineAnnealingLR (115 ep)** schedule
and **120 epochs** total. Better suited for the 12-channel model whose 9/12 input
channels start with sub-optimal pretrained weights.

**Run this only after Section 6 (v1) is complete.**


In [ ]:
import subprocess, sys
from pathlib import Path

# Pull latest repo so we get the cosine_warmup scheduler
subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--quiet'], check=True)
print('Repo up to date.')

LOCAL_MODEL_DIR_V2 = Path('/content/fusion_model_v2')
LOCAL_MODEL_DIR_V2.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable, 'scripts/train_fusion12ch.py',
        '--cache-root',      str(CACHE_ROOT),
        '--model-dir',       str(LOCAL_MODEL_DIR_V2),
        '--device',          'cuda',
        '--epochs',          '120',
        '--batch-size',      str(BATCH_SIZE),
        '--num-workers',     str(NUM_WORKERS),
        '--seed',            str(SEED),
        '--scheduler',       'cosine_warmup',
        '--warmup-epochs',   '5',
        '--mixed-precision', MIXED_PRECISION,
        '--allow-weight-download',
    ],
    check=True,
    cwd=str(REPO_DIR),
)

## 9. Save Ensemble²-v2 Artifacts to Drive

In [ ]:
import shutil, pandas as pd

MODEL_NAME_V2  = 'rgb_lab_hsv_ycrcb_cosine120_seed42'
drive_dest_v2  = DRIVE_MODEL_ROOT / 'fusion' / MODEL_NAME_V2
drive_dest_v2.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {'best_checkpoint.pt', 'training_history.csv', 'predictions.csv'}
missing = EXPECTED_FILES - {f.name for f in LOCAL_MODEL_DIR_V2.iterdir()}
if missing:
    raise RuntimeError(f'v2 training did not produce: {missing}')

for fname in EXPECTED_FILES:
    shutil.copy2(LOCAL_MODEL_DIR_V2 / fname, drive_dest_v2 / fname)
    print(f'Saved: {drive_dest_v2 / fname}')

preds = pd.read_csv(drive_dest_v2 / 'predictions.csv')
for split in ['validation', 'test']:
    mae = preds[preds['split'] == split]['absolute_error'].mean()
    print(f'  Ensemble²-v2 {split} MAE: {mae:.4f}')

print('\nDone! v2 model saved to Drive.')